# Power analysis via Monte Carlo simulation for the Attestation Trust Study.

The study uses a 3 (attestation) x 2 (correctness) x 2 (stakes) within-subjects
factorial design with crossed random effects (participant x item). The outcome
is a 7-point state-trust rating, treated as continuous for the linear
mixed-effects model.

The simulation estimates statistical power by:

1. Generating a dataset under a set of assumed fixed-effect sizes and random-effect standard deviations.
2. Fitting a mixed-effects model to that dataset.
3. Recording whether each target effect is statistically significant.
4. Repeating many times; the proportion of significant results is the power.
5. Sweeping the participant count until the target effect reaches ~0.80 power.

The output is only as reliable as the assumed effect sizes in ``BETA``. Estimate those from pilot data, then re-run to obtain a defensible recruitment target.

#### Note:
``statsmodels`` handles crossed random effects less cleanly than R's ``lme4``/``simr``. This script models the participant random intercept as the primary grouping and adds the item random intercept as a variance component. For the final reported sample size, cross-check the number in the companion ``simr`` script.

**AI assistance disclosure:** This analysis/tooling script (power simulation for sample-size determination) was generated with the assistance of an AI assistant (Claude) and reviewed, adapted, and verified by Harry Staley.
It is research tooling, not graded analytical prose or experimental stimuli. The statistical design (model specification, effect-size assumptions, interpretation of results) and all reported conclusions are the author's own. Use of generative AI follows the CS 6795 course policy.

In [ ]:
"""Monte Carlo power analysis (with effect-size sweep) for the Attestation Trust Study.

AI assistance disclosure: this simulation/plumbing utility was written with the
assistance of an AI assistant (Claude) and reviewed by the author. It implements
the power simulation; the design, hypotheses, and the choice of which effect size
to power for are the author's. This is tooling, not graded analytical content.
Use of generative AI follows the CS 6795 course policy.

WHAT THIS DOES
--------------
Estimates statistical power for the study's 3 (attestation) x 2 (correctness)
x 2 (stakes) within-subjects design by Monte Carlo simulation:

  1. SIMULATE a dataset from assumed fixed effects (BETA) + random effects (SDs).
  2. FIT the planned mixed-effects model to that simulated dataset.
  3. RECORD whether each target term (H2 two-way, H4 three-way) is significant.
  4. REPEAT N_SIMS times; power = fraction of fits in which the term is detected.
  5. SWEEP across a range of assumed H2 effect sizes (ATT_X_CORRS) and sample
     sizes (N_TO_TEST), so you can read off the N required for any effect size.

IMPORTANT: the simulation does NOT read real data. It generates synthetic data
from the parameters below. Real/pilot data informs power only by supplying good
values for those parameters (SDs here are pilot-derived; the H2 effect size is
the author's smallest-effect-of-interest, swept to show sensitivity).

HOW TO USE
----------
1. Set the SDs from your pilot (already done).
2. Set ATT_X_CORRS to the H2 effect sizes you want to consider.
3. Run. Read the per-effect-size power curves and the two summary tables.
4. Commit to ONE effect size (your smallest-effect-of-interest); report its N.

REQUIRES: numpy, pandas, statsmodels.

RUNTIME: total fits = len(ATT_X_CORRS) * len(N_TO_TEST) * N_SIMS. With the
defaults that is 7 * 13 * 500 = 45,500 fits, which can take tens of minutes. For
a quick pass, lower N_SIMS to ~200 and/or trim the lists.
"""

from __future__ import annotations

import warnings
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")  # silence MixedLM convergence chatter

# Seed: fixes the random stream so runs reproduce identical numbers. Change it
# for an independent replication; keep it fixed for a reportable run.
RNG: np.random.Generator = np.random.default_rng(6795)


# ===========================================================================
# PARAMETERS / KNOBS
# Every value below can be turned. Each is documented with what it means, what
# changing it does, and where its value should come from.
# ===========================================================================

# --- DESIGN CONSTANTS (change only if the study design changes) -------------

# Number of distinct question stems (6 low + 6 high). The item random-effect
# level count. Fixed by design; change only if you add/remove stems.
N_STEMS: int = 12


# --- EFFECT-SIZE SWEEP (the main analysis knob) -----------------------------

# H2 (attestation x correctness) effect sizes to SWEEP, as integers in hundredths
# of a trust-scale point (15 -> 0.15), divided by 100 internally. THE KEY KNOB:
# H2 is the primary hypothesis and required N depends almost entirely on this.
# Each entry produces its own power curve and summary row. Smaller -> larger N.
# Anchor your final choice in theory/literature, NOT the noisy pilot estimate.
ATT_X_CORRS: List[int] = [10, 15, 20, 25, 30, 35, 40]


# --- ASSUMED FIXED EFFECTS (the data-generating truth) ----------------------

# Fixed-effect sizes in points on the 7-point trust scale; the "true" effects the
# simulation builds data from. att_x_corr here is the BASELINE default; sweep()
# OVERRIDES it per iteration from ATT_X_CORRS. Other terms are held constant.
#   intercept    : grand-mean trust (cosmetic for power; pilot 3.94).
#   attestation  : H1 main effect per attestation level (0/1/2); pilot ~0.59.
#   correctness  : correct - incorrect main effect. Not the power target.
#   stakes       : high - low main effect. Not the power target.
#   att_x_corr   : H2, the CORE interaction and primary power target (swept).
#   att_x_stakes : nuisance two-way; not hypothesized.
#   corr_x_stakes: nuisance two-way; not hypothesized.
#   three_way    : H4 (att x corr x stakes); exploratory, hard to power.
BETA: Dict[str, float] = dict(
    intercept=3.94,        # grand-mean trust (pilot)
    attestation=0.59,      # H1 main effect per attestation level (pilot)
    correctness=0.30,      # correct - incorrect
    stakes=0.10,           # high - low
    att_x_corr=0.25,       # H2 baseline (OVERRIDDEN by the sweep)
    att_x_stakes=0.10,     # nuisance two-way
    corr_x_stakes=0.10,    # nuisance two-way
    three_way=0.15,        # H4 three-way (exploratory)
)


# --- VARIANCE COMPONENTS (random-effect SDs; set from the pilot) ------------

# Between-participant SD of the random intercept: how much participants differ in
# baseline trust. Larger -> more between-person noise -> lower power. Pilot 0.89.
SD_PARTICIPANT: float = 0.89
# Between-stem SD: how much items differ in baseline trust. Larger -> lower power.
# Pilot 0.82 (much larger than the original 0.30 guess -> stems vary more).
SD_ITEM: float = 0.82
# Trial-level residual SD: unexplained per-response noise. Larger -> lower power.
# Pilot 0.84 (lower than the original 1.10 guess -> helps power).
SD_RESIDUAL: float = 0.84


# --- SIMULATION CONTROL (precision vs. runtime) -----------------------------

# Simulated datasets per (effect size, N) cell. More -> more precise power but
# longer runtime. ~200 to explore, 500 for the full sweep, 1000 for a single
# final curve. Estimate precision ~ sqrt(p(1-p)/N_SIMS).
N_SIMS: int = 1000
# Participant counts to evaluate. Finely spaced where plausible effects cross
# 0.80 (60-200), coarser in the tail. Summary 2 reads N=80/120/160 if present.
N_TO_TEST: List[int] = [60, 80, 100, 120, 140, 160, 180, 200, 240, 280, 320, 400, 500]
# Power threshold you target (convention 0.80). Summaries flag the smallest N
# reaching this and mark cells that meet it.
TARGET_POWER: float = 0.80
# Significance threshold for declaring a term detected in each simulated fit.
ALPHA: float = 0.05
# Terms to evaluate power for: display label -> model term. H1 (main effect) and
# H2 (two-way) are confirmatory; H4 (three-way) is exploratory. All are read from
# a single fit per dataset (see fit_once).
# NOTE: H3 (reliance mediated by trust) is a MEDIATION hypothesis and is NOT
# estimable by this fixed-effect significance simulation; it requires a separate
# mediation power analysis and is intentionally absent here.
TARGET_TERMS: Dict[str, str] = {
    "H1_attestation": "attestation",
    "H2_attestation:correctness": "attestation:correctness",
    "H4_attestation:correctness:stakes": "attestation:correctness:stakes",
}
# ===========================================================================


def simulate_dataset(
    n_participants: int,
    betas: Dict[str, float] = BETA,
    sd_part: float = SD_PARTICIPANT,
    sd_item: float = SD_ITEM,
    sd_resid: float = SD_RESIDUAL,
    n_stems: int = N_STEMS,
    rng: np.random.Generator = RNG,
) -> pd.DataFrame:
    """Generate one simulated dataset for the 3x2x2 within-subjects design.

    Each participant contributes one trial per cell (12 trials). Stems rotate
    across cells by participant so item is crossed with condition (the study's
    Latin-square counterbalancing). Outcome = fixed effects + participant and
    item random intercepts + residual noise. ``betas`` is explicit so the sweep
    can vary att_x_corr while holding all else fixed.

    Args:
        n_participants: Participants to simulate (the N tested).
        betas: Fixed-effect sizes (sweep passes a modified copy).
        sd_part: Participant random-intercept SD (see :data:`SD_PARTICIPANT`).
        sd_item: Item/stem random-intercept SD (see :data:`SD_ITEM`).
        sd_resid: Trial-level residual SD (see :data:`SD_RESIDUAL`).
        n_stems: Stems to rotate across cells (see :data:`N_STEMS`).
        rng: NumPy generator for reproducibility.

    Returns:
        Long-format DataFrame: one row per participant-trial with ``participant``,
        ``item`` (categorical), ``attestation`` (0/1/2), ``correctness``
        (-0.5/0.5), ``stakes`` (-0.5/0.5), ``y`` (simulated trust).
    """
    att_levels = [0, 1, 2]        # None / Weak / Strong (linear-trend coding)
    corr_levels = [-0.5, 0.5]     # incorrect / correct (centered)
    stake_levels = [-0.5, 0.5]    # low / high (centered)
    cells: List[Tuple[int, float, float]] = [
        (a, c, s) for a in att_levels for c in corr_levels for s in stake_levels
    ]

    part_re = rng.normal(0, sd_part, size=n_participants)  # participant intercepts
    item_re = rng.normal(0, sd_item, size=n_stems)          # item intercepts

    rows: List[Tuple[int, int, int, float, float, float]] = []
    for p in range(n_participants):
        for i, (a, c, s) in enumerate(cells):
            item = (i + p) % n_stems  # rotate stem so item is crossed with cell
            mu = (
                betas["intercept"]
                + betas["attestation"] * a
                + betas["correctness"] * c
                + betas["stakes"] * s
                + betas["att_x_corr"] * a * c
                + betas["att_x_stakes"] * a * s
                + betas["corr_x_stakes"] * c * s
                + betas["three_way"] * a * c * s
                + part_re[p]
                + item_re[item]
            )
            y = mu + rng.normal(0, sd_resid)  # trial-level residual noise
            rows.append((p, item, a, c, s, y))

    df = pd.DataFrame(
        rows,
        columns=["participant", "item", "attestation", "correctness", "stakes", "y"],
    )
    df["participant"] = df["participant"].astype("category")
    df["item"] = df["item"].astype("category")
    return df


def fit_once(df: pd.DataFrame, terms: Dict[str, str]) -> Dict[str, float]:
    """Fit the mixed model ONCE and return the p-value for every target term.

    Fitting once and reading all coefficients halves compute versus fitting per
    term. Uses a participant random intercept plus an item variance component.

    Args:
        df: A dataset from :func:`simulate_dataset`.
        terms: Display label -> model term to extract.

    Returns:
        ``{label: p_value}``; ``float('nan')`` if the fit failed to converge or
        the term was not found.
    """
    vc = {"item": "0 + C(item)"}  # item variance component (crossed structure)
    model = smf.mixedlm(
        "y ~ attestation * correctness * stakes",
        data=df,
        groups=df["participant"],
        vc_formula=vc,
        re_formula="1",  # participant random intercept
    )
    try:
        res = model.fit(reml=False, method="lbfgs", maxiter=200)
    except Exception:
        return {label: float("nan") for label in terms}

    out: Dict[str, float] = {}
    for label, term in terms.items():
        t = term
        if t not in res.pvalues.index:
            # statsmodels may name interactions differently; match by factor set.
            candidates = [
                ix for ix in res.pvalues.index
                if set(ix.split(":")) == set(term.split(":"))
            ]
            t = candidates[0] if candidates else None
        out[label] = float(res.pvalues[t]) if t is not None else float("nan")
    return out


def fit_and_test(df: pd.DataFrame, term: str) -> float:
    """Backward-compatible single-term wrapper around :func:`fit_once`."""
    return fit_once(df, {"_": term})["_"]


def power_at_N(
    n_participants: int,
    betas: Dict[str, float],
    target_terms: Dict[str, str] = TARGET_TERMS,
    n_sims: int = N_SIMS,
    alpha: float = ALPHA,
) -> Tuple[Dict[str, float], Dict[str, int]]:
    """Estimate power for each target term at one sample size, using ``betas``.

    Args:
        n_participants: Participants per simulated dataset (the N tested).
        betas: Fixed-effect sizes (sweep passes a per-effect-size copy).
        target_terms: Label -> model term (see :data:`TARGET_TERMS`).
        n_sims: Simulated datasets (see :data:`N_SIMS`).
        alpha: Significance threshold (see :data:`ALPHA`).

    Returns:
        ``(power, valid)``: ``power`` = significant/converged fits per label;
        ``valid`` = converged-fit count per label (diagnostic; many failures mean
        the crossed structure is straining statsmodels).
    """
    hits: Dict[str, int] = {name: 0 for name in target_terms}
    valid: Dict[str, int] = {name: 0 for name in target_terms}

    for _ in range(n_sims):
        df = simulate_dataset(n_participants, betas=betas)
        pvals = fit_once(df, target_terms)  # ONE fit, all terms
        for name in target_terms:
            p = pvals[name]
            if not np.isnan(p):
                valid[name] += 1
                if p < alpha:
                    hits[name] += 1

    power: Dict[str, float] = {}
    for name in target_terms:
        v = valid[name]
        power[name] = hits[name] / v if v else float("nan")
    return power, valid


def power_curve(betas: Dict[str, float], n_list: List[int] = N_TO_TEST) -> pd.DataFrame:
    """Print and return a power curve (power vs. N) for one effect-size setting.

    Args:
        betas: Fixed-effect sizes for this curve (sweep sets att_x_corr).
        n_list: Sample sizes to evaluate (see :data:`N_TO_TEST`).

    Returns:
        DataFrame with column ``N`` and one column per target label (power).
    """
    print(f"{'N':>5} | " + " | ".join(f"{name:>38}" for name in TARGET_TERMS))
    print("-" * (8 + 41 * len(TARGET_TERMS)))

    results: List[Dict[str, float]] = []
    for n in n_list:
        power, valid = power_at_N(n, betas=betas)
        row: Dict[str, float] = {"N": n}
        cells: List[str] = []
        for name in TARGET_TERMS:
            pw = power[name]
            row[name] = pw
            flag = "  <-- >=80%" if (not np.isnan(pw) and pw >= TARGET_POWER) else ""
            cells.append(f"{pw:>.2f} (valid {valid[name]}/{N_SIMS}){flag}")
        print(f"{n:>5} | " + " | ".join(f"{c:>38}" for c in cells))
        results.append(row)
    return pd.DataFrame(results)


def smallest_n_for_power(curve: pd.DataFrame, term_label: str,
                         target: float = TARGET_POWER) -> int | None:
    """Return the smallest N in a curve that reaches ``target`` power, or None.

    Args:
        curve: A DataFrame from :func:`power_curve`.
        term_label: Which target column to check (e.g. the H2 label).
        target: Power threshold (see :data:`TARGET_POWER`).

    Returns:
        Smallest qualifying ``N``, or ``None`` if no tested N reaches target.
    """
    hits = curve[curve[term_label] >= target]
    return int(hits["N"].min()) if len(hits) else None


def build_results_table(curves_detail: List[Dict]) -> "pd.DataFrame":
    """Assemble a tidy long-format results table for notebook documentation.

    One row per (att_x_corr, N, hypothesis) with the estimated power, the count
    of significant fits, and the count of converged fits. This is the saveable
    artifact for documenting the analysis (renders cleanly in a notebook and
    exports to CSV).

    Args:
        curves_detail: List of per-cell dicts collected during the sweep, each
            with keys att_x_corr, N, hypothesis, power, n_significant, n_valid.

    Returns:
        A DataFrame sorted by att_x_corr, N, hypothesis.
    """
    df = pd.DataFrame(curves_detail)
    df = df.sort_values(["att_x_corr", "N", "hypothesis"]).reset_index(drop=True)
    return df


def run_and_save(out_csv: str = "power_results_detailed.csv") -> "pd.DataFrame":
    """Run the full sweep, print curves + summaries, and SAVE a detailed table.

    Produces the same console output as :func:`sweep` but also collects a tidy
    long-format table (one row per effect size x N x hypothesis) and writes it to
    ``out_csv``. Returns the table so a notebook cell can display it directly.

    Args:
        out_csv: Path to write the detailed results CSV.

    Returns:
        The tidy results DataFrame (also written to ``out_csv``).
    """
    print("=" * 78)
    print("ATTESTATION TRUST STUDY -- power sweep (detailed table mode)")
    print(f"Effect sizes swept: {[v / 100 for v in ATT_X_CORRS]}")
    print(f"{N_SIMS} sims per N, alpha={ALPHA}. SDs from pilot.")
    print("=" * 78)

    detail: List[Dict] = []
    curves: Dict[float, pd.DataFrame] = {}

    for raw in ATT_X_CORRS:
        eff = raw / 100.0
        betas = dict(BETA)
        betas["att_x_corr"] = eff
        print(f"\n{'#' * 78}")
        print(f"# att_x_corr = {eff:.2f}")
        print(f"{'#' * 78}")
        print(f"{'N':>5} | " + " | ".join(f"{name:>38}" for name in TARGET_TERMS))
        print("-" * (8 + 41 * len(TARGET_TERMS)))

        curve_rows: List[Dict[str, float]] = []
        for n in N_TO_TEST:
            # inline the per-N computation so we can capture hit/valid counts
            hits = {name: 0 for name in TARGET_TERMS}
            valid = {name: 0 for name in TARGET_TERMS}
            for _ in range(N_SIMS):
                df_sim = simulate_dataset(n, betas=betas)
                pvals = fit_once(df_sim, TARGET_TERMS)
                for name in TARGET_TERMS:
                    p = pvals[name]
                    if not np.isnan(p):
                        valid[name] += 1
                        if p < ALPHA:
                            hits[name] += 1

            row: Dict[str, float] = {"N": n}
            cells: List[str] = []
            for name in TARGET_TERMS:
                v = valid[name]
                pw = hits[name] / v if v else float("nan")
                row[name] = pw
                flag = "  <-- >=80%" if (not np.isnan(pw) and pw >= TARGET_POWER) else ""
                cells.append(f"{pw:>.2f} (valid {v}/{N_SIMS}){flag}")
                # record a tidy detail row per hypothesis
                detail.append({
                    "att_x_corr": eff,
                    "N": n,
                    "hypothesis": name,
                    "power": pw,
                    "n_significant": hits[name],
                    "n_valid": v,
                    "n_sims": N_SIMS,
                    "alpha": ALPHA,
                })
            print(f"{n:>5} | " + " | ".join(f"{c:>38}" for c in cells))
            curve_rows.append(row)
        curves[eff] = pd.DataFrame(curve_rows)

    # reuse the summary printing from sweep() by recomputing from curves
    _print_summaries(curves)

    table = build_results_table(detail)
    out_path = Path(out_csv)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(out_path, index=False)
    print(f"\nSaved detailed results table -> {out_path}  ({len(table)} rows)")
    print("Columns: att_x_corr, N, hypothesis, power, n_significant, n_valid, n_sims, alpha")
    print("Note: H3 (mediation) is not in this table -- it needs a separate")
    print("mediation power analysis and is not estimable here.")
    return table


def _print_summaries(curves: Dict[float, pd.DataFrame]) -> None:
    """Print Summary 1 (min N) and Summary 2 (power at candidate Ns)."""
    h2_label = "H2_attestation:correctness"
    h4_label = "H4_attestation:correctness:stakes"

    print("\n" + "=" * 78)
    print("SUMMARY 1: smallest N reaching {:.0%} power".format(TARGET_POWER))
    print("=" * 78)
    print(f"{'att_x_corr':>11} | {'min N (H2)':>11} | {'min N (H4)':>11}")
    print("-" * 39)
    for eff in sorted(curves):
        n2 = smallest_n_for_power(curves[eff], h2_label)
        n4 = smallest_n_for_power(curves[eff], h4_label)
        s2 = str(n2) if n2 is not None else f">{max(N_TO_TEST)}"
        s4 = str(n4) if n4 is not None else f">{max(N_TO_TEST)}"
        print(f"{eff:>11.2f} | {s2:>11} | {s4:>11}")

    spotlight_ns = [n for n in (80, 120, 160) if n in N_TO_TEST]
    print("\n" + "=" * 78)
    print("SUMMARY 2: H2 power achieved at candidate sample sizes")
    print("=" * 78)
    header = f"{'att_x_corr':>11} | " + " | ".join(f"N={n:<5}" for n in spotlight_ns)
    print(header)
    print("-" * len(header))
    for eff in sorted(curves):
        curve = curves[eff]
        cells = []
        for n in spotlight_ns:
            row = curve[curve["N"] == n]
            pw = float(row[h2_label].iloc[0]) if len(row) else float("nan")
            mark = "*" if pw >= TARGET_POWER else " "
            cells.append(f"{pw:>.2f}{mark}")
        print(f"{eff:>11.2f} | " + " | ".join(f"{c:<7}" for c in cells))
    print("  (* = meets {:.0%} target)".format(TARGET_POWER))


def sweep() -> Dict[float, pd.DataFrame]:
    """Run a full power curve for every effect size in :data:`ATT_X_CORRS`.

    Prints each effect size's full power curve, then two summaries:
      * Summary 1: smallest N reaching TARGET_POWER for H2 and H4 per effect size.
      * Summary 2: H2 power at candidate Ns (80/120/160 if in N_TO_TEST).

    Returns:
        ``{att_x_corr_value: power_curve_dataframe}`` for further inspection.
    """
    print("=" * 78)
    print("ATTESTATION TRUST STUDY -- power simulation SWEEP over att_x_corr")
    print(f"Effect sizes swept: {[v / 100 for v in ATT_X_CORRS]}")
    print(f"{N_SIMS} sims per N, alpha={ALPHA}. SDs from pilot.")
    print("=" * 78)

    curves: Dict[float, pd.DataFrame] = {}
    h2_label = "H2_attestation:correctness"
    h4_label = "H4_attestation:correctness:stakes"

    # --- full per-effect-size curves -----------------------------------------
    for raw in ATT_X_CORRS:
        eff = raw / 100.0
        betas = dict(BETA)            # copy so the global is not mutated
        betas["att_x_corr"] = eff     # override only the H2 effect size
        print(f"\n{'#' * 78}")
        print(f"# att_x_corr = {eff:.2f}")
        print(f"{'#' * 78}")
        curves[eff] = power_curve(betas)

    # --- Summary 1: smallest N for target power (H2 and H4) -------------------
    print("\n" + "=" * 78)
    print("SUMMARY 1: smallest N reaching {:.0%} power".format(TARGET_POWER))
    print("=" * 78)
    print(f"{'att_x_corr':>11} | {'min N (H2)':>11} | {'min N (H4)':>11}")
    print("-" * 39)
    for raw in ATT_X_CORRS:
        eff = raw / 100.0
        n2 = smallest_n_for_power(curves[eff], h2_label)
        n4 = smallest_n_for_power(curves[eff], h4_label)
        s2 = str(n2) if n2 is not None else f">{max(N_TO_TEST)}"
        s4 = str(n4) if n4 is not None else f">{max(N_TO_TEST)}"
        print(f"{eff:>11.2f} | {s2:>11} | {s4:>11}")

    # --- Summary 2: H2 power at spotlight Ns ----------------------------------
    spotlight_ns = [n for n in (80, 120, 160) if n in N_TO_TEST]
    print("\n" + "=" * 78)
    print("SUMMARY 2: H2 power achieved at candidate sample sizes")
    print("=" * 78)
    header = f"{'att_x_corr':>11} | " + " | ".join(f"N={n:<5}" for n in spotlight_ns)
    print(header)
    print("-" * len(header))
    for raw in ATT_X_CORRS:
        eff = raw / 100.0
        curve = curves[eff]
        cells = []
        for n in spotlight_ns:
            row = curve[curve["N"] == n]
            pw = float(row[h2_label].iloc[0]) if len(row) else float("nan")
            mark = "*" if pw >= TARGET_POWER else " "
            cells.append(f"{pw:>.2f}{mark}")
        print(f"{eff:>11.2f} | " + " | ".join(f"{c:<7}" for c in cells))
    print("  (* = meets {:.0%} target)".format(TARGET_POWER))

    print("\nPick the effect size that matches your smallest-effect-of-interest;")
    print("the corresponding min N (Summary 1) is your target. Summary 2 shows")
    print("how much power you buy at common Ns. H4 (three-way) is exploratory.")
    return curves


if __name__ == "__main__":
    # run_and_save() prints the curves + summaries AND writes the detailed
    # per-(effect size, N, hypothesis) table to CSV for notebook documentation.
    # Use sweep() instead if you only want the console output without the table.
    results_table = run_and_save("power_results_detailed.csv")
    # In a notebook, display the table with:  results_table

ATTESTATION TRUST STUDY -- power sweep (detailed table mode)
Effect sizes swept: [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4]
1000 sims per N, alpha=0.05. SDs from pilot.

##############################################################################
# att_x_corr = 0.10
##############################################################################
    N |                         H1_attestation |             H2_attestation:correctness |      H4_attestation:correctness:stakes
-----------------------------------------------------------------------------------------------------------------------------------
   60 |      1.00 (valid 1000/1000)  <-- >=80% |                 0.09 (valid 1000/1000) |                 0.05 (valid 1000/1000)
   80 |      1.00 (valid 1000/1000)  <-- >=80% |                 0.12 (valid 1000/1000) |                 0.07 (valid 1000/1000)
  100 |      1.00 (valid 1000/1000)  <-- >=80% |                 0.17 (valid 1000/1000) |                 0.08 (valid 1000/1000)
  120 | 